# 卖家第三步：卖家汇总与多卖家订单
第十五份 Notebook 的三题已经核对。你已理解：补入卖家属性后，输出仍是一行一条商品明细。
本课用 groupby + agg 将销售明细汇总为一行一个卖家。

| 工作表 | 一行代表什么 | 主键 |
| --- | --- | --- |
| items_with_sellers | 一笔订单中的一条商品明细，附卖家属性 | order_id + order_item_id |
| seller_summary | 一个卖家的销售汇总 | seller_id |
| order_seller_pairs | 某个订单与某个卖家的关系及明细数 | order_id + seller_id |
| seller_profile | 一个卖家的销售汇总，补地区属性 | seller_id |

**本课目标：选对统计函数，并分清哪些数量可以跨卖家相加。**
本课仍使用全量商品明细，没有筛选订单状态或日期；这里只建立销售基线，尚未评价卖家绩效。


## 1. 独立读取原表
本课复用商品明细清洗函数，金额始终由原始字符串精确换成整数分。
下面先复现前课的关联，后面的汇总不依赖另一份 Notebook 的内存。


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从 olist-operations-analytics 项目目录打开本课')

data_dir = project_dir.parent / 'archive'
src_dir = str(project_dir / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from items_cleaning import clean_order_items
from customers_cleaning import STATE_CODES

def read_source(filename):
    return pd.read_csv(
        data_dir / filename, dtype=str, encoding='utf-8-sig',
        keep_default_na=False, na_values=['']
    )

sellers_raw = read_source('olist_sellers_dataset.csv')
items_raw = read_source('olist_order_items_dataset.csv')
items_clean = clean_order_items(items_raw)

print('卖家属性记录数：', len(sellers_raw))
print('规范后商品明细数：', len(items_clean))


卖家属性记录数： 3095
规范后商品明细数： 112650


## 2. 复现卖家工作副本及明细关联
保留原始值，在副本中规范地区格式并保留核查标记。
与上一课相同：先检查卖家主键和明细覆盖，再做多对一左连接。


In [2]:
seller_ids = sellers_raw['seller_id']
if (
    seller_ids.isna().any()
    or seller_ids.str.strip().eq('').any()
    or seller_ids.duplicated().any()
):
    raise ValueError('卖家主键异常，请先核查')

sellers_clean = sellers_raw.copy()
for col in ['seller_zip_code_prefix', 'seller_city', 'seller_state']:
    sellers_clean[col] = sellers_clean[col].str.strip()
sellers_clean['seller_city'] = sellers_clean['seller_city'].str.lower()
sellers_clean['seller_state'] = sellers_clean['seller_state'].str.upper()
sellers_clean['flag_zip_format_invalid'] = (
    ~sellers_clean['seller_zip_code_prefix'].str.fullmatch('[0-9]{5}', na=False)
)
sellers_clean['flag_state_format_invalid'] = (
    ~sellers_clean['seller_state'].str.fullmatch('[A-Z]{2}', na=False)
)
sellers_clean['flag_state_unknown'] = ~sellers_clean['seller_state'].isin(STATE_CODES)
sellers_clean['flag_city_missing'] = (
    sellers_clean['seller_city'].isna() | sellers_clean['seller_city'].eq('')
)

if (~items_clean['seller_id'].isin(sellers_clean['seller_id'])).any():
    raise ValueError('存在找不到卖家属性的销售明细，请先核查')

items_with_sellers = items_clean.merge(
    sellers_clean, on='seller_id', how='left',
    validate='many_to_one', indicator='sellers_match'
)

# 关联后，明细主键和数量继续保持原粒度。
if (
    len(items_with_sellers) != len(items_clean)
    or items_with_sellers.duplicated(['order_id', 'order_item_id']).any()
):
    raise ValueError('关联后明细数量或主键异常')

# 核对金额完整且与来源相等，再进入本课的卖家聚合。
money_cols = ['price_cents', 'freight_cents']
if items_with_sellers[money_cols].isna().any().any():
    raise ValueError('明细金额缺失，请先核查')
if not items_with_sellers[money_cols].sum().equals(items_clean[money_cols].sum()):
    raise ValueError('关联前后金额不一致，请先核查')

print('关联后明细行数：', len(items_with_sellers))


关联后明细行数： 112650


## 3. 核心步骤：一行一个卖家的汇总
这一步才真正减少行数，完整明细表仍保留。

| 指标 | 统计方法 | 含义 |
| --- | --- | --- |
| item_line_count | size | 该卖家有多少条商品明细 |
| order_count | order_id 的 nunique | 该卖家涉及多少个不同订单 |
| product_count | product_id 的 nunique | 该卖家销售过多少个不同商品 ID |
| goods_amount_cents | price_cents 的 sum | 该卖家明细中的商品金额，整数分 |
| freight_amount_cents | freight_cents 的 sum | 该卖家明细中的运费，整数分 |

同一卖家在一个订单里有三条明细：明细数计 3，订单数计 1。
同一商品 ID 在多条明细中出现时，product_count 在该卖家内只计一次。


In [3]:
# groupby 确定“每个卖家一组”；agg 决定每组要计算哪些指标。
# 输出字段名 = (来源字段, 聚合方法)。
seller_summary = (
    items_with_sellers
    .groupby('seller_id', as_index=False)
    .agg(
        item_line_count=('order_item_id', 'size'),
        order_count=('order_id', 'nunique'),
        product_count=('product_id', 'nunique'),
        goods_amount_cents=('price_cents', 'sum'),
        freight_amount_cents=('freight_cents', 'sum')
    )
)

# 主键随粒度变化：现在用 seller_id 检查唯一性。
if seller_summary['seller_id'].isna().any() or seller_summary['seller_id'].duplicated().any():
    raise ValueError('卖家汇总主键异常')

print('卖家汇总行数：', len(seller_summary))
display(seller_summary.head())


卖家汇总行数： 3095


,seller_id,item_line_count,order_count,product_count,goods_amount_cents,freight_amount_cents
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,1,268500,6306
1,001cca7ae9ae17fb1caed9dfb1094831,239,200,11,2508003,885414
2,001e6ad469a905060d959994f1b41e4f,1,1,1,25000,1794
3,002100f778ceb8431b7a1020ff7ab48f,55,51,24,123450,79366
4,003554e2dce176b5555353e4f3555ac8,1,1,1,12000,1938


## 4. 汇总前后控制总额
汇总表只有 3,095 行，但每行保存该卖家的明细数量和金额。
因此用 item_line_count 求和核对来源明细数，不能拿汇总表行数与来源明细数直接比较。

每条明细只归属一个 seller_id，所以明细数、商品金额和运费可跨卖家相加。
这里暂不把各卖家的不同订单数与全局不同订单数直接判作相等。


In [4]:
seller_control = pd.DataFrame({
    '来源明细': [
        len(items_with_sellers),
        items_with_sellers['price_cents'].sum(),
        items_with_sellers['freight_cents'].sum()
    ],
    '卖家汇总': [
        seller_summary['item_line_count'].sum(),
        seller_summary['goods_amount_cents'].sum(),
        seller_summary['freight_amount_cents'].sum()
    ]
}, index=['商品明细行数', '商品金额（分）', '运费（分）'])

seller_control['差额'] = seller_control['卖家汇总'] - seller_control['来源明细']
if seller_control['差额'].ne(0).any():
    raise ValueError('卖家聚合的来源控制总额不一致，请核查')

display(seller_control)


,来源明细,卖家汇总,差额
商品明细行数,112650,112650,0
商品金额（分）,1359164370,1359164370,0
运费（分）,225190954,225190954,0


## 5. 为什么卖家订单数合计可能更大
先看一个小例子：

| 订单 | 涉及卖家 | 在各卖家订单数中如何计数 |
| --- | --- | --- |
| A | 甲、乙 | 甲计 1，乙计 1 |
| B | 甲 | 甲计 1 |

全体只有 2 个不同订单，但甲的订单数为 2，乙为 1，相加为 3。
这是每个卖家参与的订单数之和，不能直接当成平台不同订单数。

下面从完整明细生成订单＋卖家关系表：同一订单、同一卖家的多条明细聚成一行，
item_line_count 保留这组有多少条明细。它是新的派生表，完整明细仍保留。


In [5]:
# 这张表一行一个“订单+卖家”组合，没有丢弃完整明细表。
order_seller_pairs = (
    items_with_sellers
    .groupby(['order_id', 'seller_id'], as_index=False)
    .agg(item_line_count=('order_item_id', 'size'))
)

# 每个订单涉及多少个不同卖家。
order_seller_counts = (
    order_seller_pairs.groupby('order_id')['seller_id'].nunique()
)

global_order_count = items_with_sellers['order_id'].nunique()
seller_order_count_sum = seller_summary['order_count'].sum()
multi_seller_order_count = order_seller_counts.gt(1).sum()

# 一个订单有 k 个卖家，会产生 k 对“订单+卖家”关系；
# 与该订单全局只计1次相比，多出的关系数是 k-1。
extra_pair_count = (order_seller_counts - 1).sum()

display(pd.Series({
    '全体不同订单数': global_order_count,
    '各卖家订单数之和': seller_order_count_sum,
    '订单+卖家关系数': len(order_seller_pairs),
    '多卖家订单数': multi_seller_order_count,
    '多出的订单+卖家关系数': extra_pair_count,
    '单个订单最多卖家数': order_seller_counts.max()
}))

# 两种口径分别核对，不要求100010与98666相等。
if seller_order_count_sum != len(order_seller_pairs):
    raise ValueError('卖家订单数合计与订单卖家关系数不一致')
if len(order_seller_pairs.groupby('order_id')) != global_order_count:
    raise ValueError('订单卖家关系表覆盖的订单数不一致')
if order_seller_pairs['item_line_count'].sum() != len(items_with_sellers):
    raise ValueError('订单卖家关系表的明细数控制不一致')
if seller_order_count_sum - global_order_count != extra_pair_count:
    raise ValueError('订单数差异与额外关系数不一致')

# 观察每单1、2、3、4、5个卖家分别有多少订单。
seller_count_distribution = (
    order_seller_counts.value_counts().sort_index()
    .rename_axis('每单卖家数').reset_index(name='订单数')
)
display(seller_count_distribution)


全体不同订单数         98666
各卖家订单数之和       100010
订单+卖家关系数       100010
多卖家订单数           1278
多出的订单+卖家关系数      1344
单个订单最多卖家数           5
dtype: int64

,每单卖家数,订单数
0,1,97388
1,2,1219
2,3,54
3,4,3
4,5,2


## 6. 回到真实订单看两种计数
选择一笔有两个卖家的订单。对全局不同订单数贡献 1，
对两个卖家各自的订单数各贡献 1，因此卖家订单数合计贡献 2。

不同商品数也要注意去重范围：product_count 在每个卖家内部去重；
同一商品 ID 可由多个卖家销售，不能默认其总和等于全局不同商品数。


In [6]:
# 先按条件找样本订单，再筛选显示它的完整明细和订单卖家关系。
example_order_id = order_seller_counts.loc[
    order_seller_counts.eq(2)
].index[0]

display(items_with_sellers.loc[
    items_with_sellers['order_id'].eq(example_order_id),
    ['order_id', 'order_item_id', 'product_id', 'seller_id', 'price_cents', 'freight_cents']
])
display(order_seller_pairs.loc[
    order_seller_pairs['order_id'].eq(example_order_id)
])

# 展示商品数的两种去重范围；差异不是商品金额控制失败。
display(pd.Series({
    '全局不同商品数': items_with_sellers['product_id'].nunique(),
    '各卖家不同商品数之和': seller_summary['product_count'].sum()
}))


,order_id,order_item_id,product_id,seller_id,price_cents,freight_cents
80,002f98c0f7efd42638ed6100ca699b42,1,d41dc2f2979f52d75d78714b378d4068,7299e27ed73d2ad986de7f7c77d919fa,899,3257
81,002f98c0f7efd42638ed6100ca699b42,2,880be32f4db1d9f6e2bec38fb6ac23ab,fa40cc5b934574b62717c68f3d678b6d,4490,716


,order_id,seller_id,item_line_count
73,002f98c0f7efd42638ed6100ca699b42,7299e27ed73d2ad986de7f7c77d919fa,1
74,002f98c0f7efd42638ed6100ca699b42,fa40cc5b934574b62717c68f3d678b6d,1


全局不同商品数       32951
各卖家不同商品数之和    34448
dtype: int64

## 7. 将卖家属性补回卖家汇总
seller_summary 已是一行一个卖家，sellers_clean 也是一行一个卖家。
这次关联可以验证 one_to_one，不再是明细关联时的 many_to_one。

返回的 seller_profile 仍一行一个卖家，供后续按地区观察卖家分布。
销售规模只是基线，当前没有计算平台利润、卖家差评率或绩效结论。


In [7]:
seller_profile = seller_summary.merge(
    sellers_clean, on='seller_id', how='left',
    validate='one_to_one', indicator='seller_attributes_match'
)

if (
    len(seller_profile) != len(seller_summary)
    or not seller_profile['seller_attributes_match'].eq('both').all()
):
    raise ValueError('卖家汇总关联属性异常')

# 聚合指标在补属性后仍保持原值。
if not seller_profile[seller_summary.columns].equals(seller_summary):
    raise ValueError('补入卖家属性后销售汇总发生变化')

display(seller_profile[[
    'seller_id', 'seller_city', 'seller_state',
    'item_line_count', 'order_count', 'product_count',
    'goods_amount_cents', 'freight_amount_cents'
]].head())
print('卖家画像基线行数：', len(seller_profile))


,seller_id,seller_city,seller_state,item_line_count,order_count,product_count,goods_amount_cents,freight_amount_cents
0,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP,3,3,1,268500,6306
1,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES,239,200,11,2508003,885414
2,001e6ad469a905060d959994f1b41e4f,sao goncalo,RJ,1,1,1,25000,1794
3,002100f778ceb8431b7a1020ff7ab48f,franca,SP,55,51,24,123450,79366
4,003554e2dce176b5555353e4f3555ac8,goiania,GO,1,1,1,12000,1938


卖家画像基线行数： 3095


## 我的理解
运行后，把卖家控制表与订单数检查结果发来，并补充：

1. seller_summary 的一行与主键是什么？item_line_count 为什么用 size，order_count 为什么用 nunique？
`seller_summary` 一行是一个卖家的销售汇总，主键是 seller_id。`size` 数这个卖家有多少条明细，`nunique` 数这些明细里有多少个不同的 order_id。
2. 为什么明细数量、商品金额、运费控制表都为零，但各卖家订单数合计为 100,010，全局不同订单数为 98,666？
因为一个订单可能有多个卖家，所以卖家订单数总数大于订单数。
3. 为什么多卖家订单有 1,278 笔，而多出的订单＋卖家关系有 1,344 对？结合每单卖家数分布解释。
单卖家订单在两种口径下都计 1 次，对差额的贡献是 0。 一笔订单如果有 3 个卖家，在全局订单数里计 1 次，在各卖家的订单数里合计 3 次，所以这一笔订单就产生 2 对额外关系。
理解之后，再把卖家阶段整理为公共函数和独立导出流程。本课只在内存处理，不覆盖原始 CSV 或前面的 Notebook。
